In [14]:
# importa bibliotecas para dados e ambiente
import os, warnings
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from dotenv import load_dotenv

# configura execucao sem interface grafica e remove avisos
matplotlib.use("Agg")
warnings.filterwarnings("ignore")

# carrega variaveis de ambiente
load_dotenv()

# define caminhos de entrada e saida apontando direto para outputs existente
CSV_DIR      = os.getenv("CSV_DIR", r"C:\Users\dcamargos\Desktop\aws-etl-assessment-agromercantil\inputs\csv")
DIR_GRAFICOS = os.getenv("DIR_GRAFICOS", r"C:\Users\dcamargos\Desktop\aws-etl-assessment-agromercantil\outputs")

In [15]:
# define lista de cores para uso em graficos
paleta = ["#378ADD"
          ,"#1D9E75"
          ,"#D85A30"
          ,"#BA7517"
          ,"#7F77DD"
          ,"#D4537E"
          ,"#639922"
          ,"#E24B4A"
]

In [16]:
# mapeia nomes de colunas para padronizacao
renomear = {
    "data_referencia": "dt_ref"
    ,"valor_brl": "val_brl"
    ,"variacao_diaria_pct": "pct_var_dia"
    ,"valor_usd": "val_usd"
}

# salva figura no diretorio definido
def salvar(fig, nome):
    caminho = os.path.join(DIR_GRAFICOS, nome)
    fig.savefig(caminho, dpi=150, bbox_inches="tight")
    plt.close(fig)

# gera lista de cores baseada na quantidade de itens
def cores_para(cultivos):
    n = len(cultivos)
    return (paleta * ((n // len(paleta)) + 1))[:n]

# aplica estilo padrao aos graficos
def estilo(fig, ax):
    fig.patch.set_facecolor("#FAFAFA")
    ax.set_facecolor("#F5F5F2")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(linestyle="--", alpha=0.3)

In [17]:
# le arquivos csv do diretorio e consolida em um unico dataframe
def carregar_dados():
    arquivos = [f for f in os.listdir(CSV_DIR) if f.endswith(".csv")]
    
    # valida existencia de arquivos
    if not arquivos:
        raise FileNotFoundError(f"nenhum .csv encontrado em: {CSV_DIR}")

    partes = []
    for arquivo in arquivos:
        caminho = os.path.join(CSV_DIR, arquivo)
        
        # extrai nome do cultivo a partir do nome do arquivo
        cultivo = os.path.splitext(arquivo)[0].replace("raw_", "").split("_")[0]
        
        # le csv e adiciona coluna de identificacao
        df_tmp  = pd.read_csv(caminho, sep=";", encoding="utf-8", on_bad_lines="skip", engine="python")
        df_tmp["cultivo"] = cultivo
        partes.append(df_tmp)

    # concatena dados e padroniza nomes de colunas
    df = pd.concat(partes, ignore_index=True).rename(columns=renomear)

    # trata coluna de data
    if "dt_ref" in df.columns:
        df["dt_ref"] = pd.to_datetime(df["dt_ref"], dayfirst=True, errors="coerce")

    # converte colunas numericas
    for col in ("val_brl", "pct_var_dia", "val_usd"):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # remove registros invalidos
    df = df.dropna(subset=["val_brl"])

    return df

In [18]:
df = carregar_dados()
df.head()

,dt_ref,val_brl,pct_var_dia,val_usd,local,data_extracao,cultivo
0,2026-03-19,100.51,2.39,1.95,São Paulo,2026-03-23 10:05:06,acucar
1,2026-03-18,98.16,1.08,-0.44,São Paulo,2026-03-23 10:05:06,acucar
2,2026-03-17,97.11,-0.52,-1.50,São Paulo,2026-03-23 10:05:06,acucar
3,2026-03-16,97.62,0.39,-0.98,São Paulo,2026-03-23 10:05:06,acucar
4,2026-03-13,97.24,1.51,-1.37,São Paulo,2026-03-23 10:05:06,acucar


In [19]:
# calcula estatisticas descritivas por cultivo
def calcular_estatisticas(df):
    stats = (
        df.groupby("cultivo")["val_brl"]
        .agg(contagem="count", media="mean", mediana="median", desvio_pad="std",
             minimo="min", maximo="max",
             q1=lambda x: x.quantile(0.25), q3=lambda x: x.quantile(0.75))
        .round(2).reset_index()
    )
    
    # calcula metricas adicionais de dispersao
    stats["amplitude_iqr"] = (stats["q3"] - stats["q1"]).round(2)
    stats["cv_pct"] = (stats["desvio_pad"] / stats["media"] * 100).round(2)
    
    return stats

In [20]:
statis = calcular_estatisticas(df)

caminho_stats = os.path.join(DIR_GRAFICOS, "estatisticas_descritivas.csv")
statis.to_csv(caminho_stats, sep=";", index=False, encoding="utf-8")

statis

,cultivo,contagem,media,mediana,desvio_pad,minimo,maximo,q1,q3,amplitude_iqr,cv_pct
0,acucar,30,103.70,101.41,6.48,95.79,117.90,98.18,109.15,10.97,6.25
1,algodao,21,361.87,363.56,7.81,348.32,374.45,355.28,367.41,12.13,2.16
2,arroz,15,57.54,57.45,1.46,55.46,60.00,56.22,58.53,2.31,2.54
3,bezerro,15,3256.09,3257.05,10.43,3225.73,3267.63,3253.15,3264.14,10.99,0.32
4,boi,36,349.93,349.94,2.86,346.05,357.41,347.38,351.65,4.27,0.82
5,cafe,30,1464.34,1457.89,443.09,989.60,1962.67,1019.80,1906.29,886.49,30.26
6,citros,45,37.22,43.10,13.81,0.00,50.25,31.25,44.35,13.10,37.10
7,frango,30,7.02,7.06,0.10,6.81,7.17,7.02,7.09,0.07,1.42
8,milho,15,71.04,71.20,0.77,69.75,72.10,70.40,71.60,1.20,1.08
9,soja,30,125.65,125.26,3.77,120.66,131.18,122.29,129.34,7.05,3.00


In [21]:
# identifica outliers por cultivo usando diferentes metodos
def detectar_outliers(df):
    resultados = []
    
    for cultivo, grupo in df.groupby("cultivo"):
        # prepara serie e calcula medidas base
        serie= grupo["val_brl"].dropna()
        q1, q3= serie.quantile(0.25), serie.quantile(0.75)
        iqr= q3 - q1
        media, std = serie.mean(), serie.std()

        # aplica metodos de deteccao de outliers
        for metodo, mask, lim_inf, lim_sup in [
            ("IQR",     (serie < q1 - 1.5*iqr) | (serie > q3 + 1.5*iqr), round(q1 - 1.5*iqr, 2), round(q3 + 1.5*iqr, 2)),
            ("Z-score", (serie - media).abs() / (std + 1e-9) > 3, round(media - 3*std, 2), round(media + 3*std, 2)),
        ]:
            # filtra registros fora dos limites e adiciona contexto
            out = grupo.loc[mask.index[mask]].copy()
            out["metodo"], out["lim_inf"], out["lim_sup"] = metodo, lim_inf, lim_sup
            resultados.append(out)

    # consolida resultados
    return pd.concat(resultados, ignore_index=True) if resultados else pd.DataFrame()

In [22]:
# executa deteccao de outliers
df_out = detectar_outliers(df)

# valida se houve resultados
if df_out.empty:
    print("nenhum outlier encontrado.")
else:
    # gera resumo por cultivo e metodo
    resumo= df_out.groupby(["cultivo", "metodo"]).size().reset_index(name="qtd_outliers")
    display(resumo)

    # salva resultado em arquivo
    caminho_out= os.path.join(DIR_GRAFICOS, "outliers_detectados.csv")
    df_out.to_csv(caminho_out, sep=";", index=False, encoding="utf-8")

,cultivo,metodo,qtd_outliers
0,bezerro,IQR,1
1,citros,IQR,4
2,frango,IQR,6
3,suino,IQR,2


In [23]:
# gera boxplot por cultivo para visualizar distribuicao de valores
def grafico_boxplot(df):
    cultivos = sorted(df["cultivo"].unique())
    dados= [df.loc[df["cultivo"] == c, "val_brl"].dropna().values for c in cultivos]

    # cria figura e aplica estilo
    fig, ax = plt.subplots(figsize=(max(10, len(cultivos) * 1.5), 6))
    estilo(fig, ax)

    # monta boxplot com configuracoes visuais
    bp = ax.boxplot(dados, patch_artist=True, notch=False, vert=True, widths=0.5
                    ,medianprops=dict(color="white", linewidth=2.5)
                    ,whiskerprops=dict(linewidth=1.2, linestyle="--", color="#888")
                    ,capprops=dict(linewidth=1.5, color="#888")
                    ,flierprops=dict(marker="o", markersize=4, alpha=0.5, linestyle="none"))

    # aplica cores nos elementos do grafico
    for patch, flier, cor in zip(bp["boxes"], bp["fliers"], cores_para(cultivos)):
        patch.set_facecolor(cor); patch.set_alpha(0.82)
        flier.set_markerfacecolor(cor); flier.set_markeredgecolor(cor)

    # ajusta rotulos e layout
    ax.set_xticks(range(1, len(cultivos) + 1))
    ax.set_xticklabels(cultivos, rotation=30, ha="right", fontsize=10)
    ax.set_ylabel("Valor BRL (R$)", fontsize=11)
    ax.set_title("Distribuição de preços por commodity — boxplot", fontsize=13, pad=14)
    ax.grid(axis="y", linestyle="--", alpha=0.4)

    plt.tight_layout()
    
    # salva grafico
    salvar(fig, "boxplot_commodities.png")

grafico_boxplot(df)

In [24]:
# gera histogramas por cultivo para analisar distribuicao de frequencia
def grafico_histogramas(df):
    cultivos= sorted(df["cultivo"].unique())
    n= len(cultivos)
    
    # define grade de subplots dinamicamente
    cols= min(4, n)
    rows= (n + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4.5, rows * 3.5))
    fig.patch.set_facecolor("#FAFAFA")
    axes_flat = np.array(axes).flatten()

    # cria um histograma para cada cultivo
    for i, (cultivo, cor) in enumerate(zip(cultivos, cores_para(cultivos))):
        ax   = axes_flat[i]
        data = df.loc[df["cultivo"] == cultivo, "val_brl"].dropna()
        
        ax.hist(data, bins=20, color=cor, alpha=0.80, edgecolor="white", linewidth=0.5)
        
        # adiciona linhas de media e mediana
        ax.axvline(data.mean(),   color="#333", linewidth=1.5, linestyle="--", label=f"média {data.mean():.1f}")
        ax.axvline(data.median(), color="#888", linewidth=1.5, linestyle=":",  label=f"mediana {data.median():.1f}")
        
        ax.set_title(cultivo, fontsize=11, pad=6)
        ax.set_xlabel("R$", fontsize=9); ax.set_ylabel("frequência", fontsize=9)
        ax.legend(fontsize=8, framealpha=0.5)

        # aplica estilo visual
        ax.set_facecolor("#F5F5F2")
        ax.spines[["top", "right"]].set_visible(False)
        ax.grid(axis="y", linestyle="--", alpha=0.3)

    # oculta eixos nao utilizados
    for j in range(i + 1, len(axes_flat)):
        axes_flat[j].set_visible(False)

    fig.suptitle("histogramas de preço BRL por commodity", fontsize=14, y=1.01)
    plt.tight_layout()
    
    # salva grafico
    salvar(fig, "histogramas_commodities.png")

grafico_histogramas(df)

In [25]:
# gera grafico de dispersao entre valor e variacao diaria
def grafico_scatter(df):
    # valida existencia da coluna necessaria
    if "pct_var_dia" not in df.columns:
        print("coluna pct_var_dia não encontrada — scatter ignorado.")
        return

    cultivos = sorted(df["cultivo"].unique())
    mapa_cor = dict(zip(cultivos, cores_para(cultivos)))

    # cria figura e aplica estilo
    fig, ax = plt.subplots(figsize=(10, 6))
    estilo(fig, ax)

    # plota pontos por cultivo
    for cultivo, cor in mapa_cor.items():
        sub = df[df["cultivo"] == cultivo]
        ax.scatter(sub["val_brl"], sub["pct_var_dia"], label=cultivo,
                   color=cor, alpha=0.55, s=28, edgecolor="none")

    # adiciona linha de referencia
    ax.axhline(0, color="#aaa", linewidth=0.8, linestyle="--")

    # ajusta rotulos e legenda
    ax.set_xlabel("Valor BRL (R$)", fontsize=11)
    ax.set_ylabel("Variação diária (%)", fontsize=11)
    ax.set_title("Preço BRL × variação diária por commodity — scatter", fontsize=13, pad=14)
    ax.legend(title="commodity", fontsize=9, title_fontsize=9, loc="upper right", framealpha=0.7)

    plt.tight_layout()
    
    # salva grafico
    salvar(fig, "scatter_brl_variacao.png")

grafico_scatter(df)

In [26]:
# gera grafico de serie temporal por cultivo
def grafico_serie_temporal(df):
    # valida existencia da coluna de data
    if "dt_ref" not in df.columns:
        print("coluna dt_ref não encontrada — série temporal ignorada.")
        return

    cultivos = sorted(df["cultivo"].unique())

    # cria figura e aplica estilo
    fig, ax = plt.subplots(figsize=(12, 5))
    estilo(fig, ax)

    # calcula media por data e plota linhas por cultivo
    for cultivo, cor in zip(cultivos, cores_para(cultivos)):
        sub = (df[df["cultivo"] == cultivo]
               .groupby("dt_ref")["val_brl"].mean()
               .reset_index().sort_values("dt_ref"))
        ax.plot(sub["dt_ref"], sub["val_brl"], label=cultivo, color=cor, linewidth=1.6, alpha=0.85)

    # ajusta rotulos e legenda
    ax.set_xlabel("data", fontsize=11)
    ax.set_ylabel("Valor BRL médio (R$)", fontsize=11)
    ax.set_title("Evolução de preços por commodity — série temporal", fontsize=13, pad=14)
    ax.legend(title="commodity", fontsize=9, title_fontsize=9, loc="upper left", framealpha=0.7)

    plt.tight_layout()
    
    # salva grafico
    salvar(fig, "serie_temporal_commodities.png")

grafico_serie_temporal(df)